# Job Interview Guide — Personalized ML Workshop
**ML Specialist interview | Post-assessment learning lab**

This notebook targets your five missed questions: **Q1 (supervised vs. unsupervised), Q3 (train/validation/test), Q6 (KNN `k`), Q9 (preprocessing leakage), and Q12 (classification thresholds)**.

### Learning workflow
Follow the same pattern as the supplied workshop notebooks: **concept → small real-world example → inspect/visualize → exercise TODO → sanity checks → reflection**. This notebook uses in-memory data and standard scikit-learn datasets: no downloaded files, external project modules, or database credentials required.

**Learning objectives:** distinguish clustering from classification; separate model selection from final evaluation; tune `k` on validation data; fit transformers only on training data; reason about threshold-dependent metrics.

> **Important:** Your other ten answers were correct. They are not the focus of this remedial notebook. This is a practice notebook, not a substitute for a full project evaluation.

## 0. Setup and reproducibility
**Libraries:** NumPy, pandas, matplotlib, scikit-learn. Optionally create a virtual environment and install dependencies with `pip install numpy pandas matplotlib scikit-learn notebook`.

Reuse a fixed random seed, inspect data types and shapes, and build sanity checks before trying more complicated models.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_classification
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Imports successful; random_state =', RANDOM_STATE)

## 1. Supervised vs. unsupervised learning (missed Q1)
**When the target label exists**, supervised classification can learn a mapping from features `X` to class labels `y`. **When the dataset has no known segment labels**, clustering can look for groups of similar observations without claiming those groups are verified labels.

**Retail example:** customers are described by purchase frequency and typical basket value. Marketing wants segments but has never assigned segment labels. K-Means produces **cluster IDs**, not verified churn/fraud outcomes. Be cautious: results depend on scaling, chosen cluster count, and the suitability of the cluster geometry.

The first experiment deliberately uses `make_blobs` to simulate customer groups and discards its generated labels for fitting.

In [ ]:
X_customers, _simulation_labels = make_blobs(n_samples=180, centers=3, cluster_std=1.1, random_state=RANDOM_STATE)
customer_df = pd.DataFrame(X_customers, columns=['purchase_frequency_index', 'basket_value_index'])
print(customer_df.head())
segments = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10).fit_predict(customer_df)
assert segments.shape == (180,)
assert len(np.unique(segments)) == 3
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(customer_df.iloc[:, 0], customer_df.iloc[:, 1], c=segments, cmap='viridis', s=24)
ax.set(xlabel='Purchase frequency index', ylabel='Basket value index', title='Exploratory customer segments (not verified labels)')
plt.show()

### Exercise 1 — choose the learning task
**TODOs:** (a) Write the target variable for a hypothetical dataset with verified churn labels. (b) Change the number of clusters to two and observe the difference. (c) Explain why the cluster IDs should not be called ground-truth churn classes.

In [ ]:
# TODO: Record whether each project is classification, regression, or clustering.
project_tasks = {
    'discover customer groups without labels': None,  # TODO
    'predict verified churn (yes/no)': None,           # TODO
    'predict a numerical rental price': None,         # TODO
}
# TODO: Try n_clusters=2 and inspect how the assignments change.
# two_cluster_labels = KMeans(n_clusters=..., random_state=RANDOM_STATE, n_init=10).fit_predict(customer_df)
# assert len(np.unique(two_cluster_labels)) == 2
print('Complete project_tasks, then add an assertion to validate your choices.')

## 2. Train, validation, and test (missed Q3)
Use the **training set** to estimate model parameters, the **validation set** to compare choices and tune hyperparameters, and a **held-out test set** for a final, ideally one-time estimate of generalization. Hyperparameter search against test results leaks knowledge about the test set into the design process even if the test labels are never included in `.fit()`.

For real subscription or failure forecasting, consider **time-ordered splits** when future records must be predicted from the past. For the independent synthetic example below, a stratified random split is appropriate.

Splitting twice below yields 60% train, 20% validation, 20% test.

In [ ]:
X, y = make_classification(n_samples=1000, n_features=6, n_informative=4, n_redundant=0,
                           class_sep=1.0, random_state=RANDOM_STATE)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.40,
                                                    stratify=y, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50,
                                               stratify=y_temp, random_state=RANDOM_STATE)
print('Train / validation / test:', len(y_train), len(y_val), len(y_test))
assert (len(y_train), len(y_val), len(y_test)) == (600, 200, 200)
assert len(y_train) + len(y_val) + len(y_test) == len(y)

### Exercise 2 — model selection discipline
Tune the model using **validation accuracy only**, then evaluate the selected model on `X_test` once. **TODO:** fill in the candidate values and best-model selection; add a note about why looking at test scores after every trial is a problem.

In [ ]:
# Demonstration: candidate selection using validation, NOT test.
results = []
for c in [0.01, 0.1, 1.0, 10.0]:
    model = Pipeline([('scale', StandardScaler()), ('classifier', LogisticRegression(C=c, max_iter=1000))])
    model.fit(X_train, y_train)
    results.append((c, accuracy_score(y_val, model.predict(X_val))))
print(pd.DataFrame(results, columns=['C', 'validation_accuracy']))
best_c = max(results, key=lambda item: item[1])[0]
print('Chosen C based on validation:', best_c)
# TODO: Fit a fresh pipeline with best_c using training + validation data if appropriate.
# TODO: Evaluate on X_test ONCE when you finish model selection.
# final_test_accuracy = ...
# assert 0 <= final_test_accuracy <= 1

## 3. KNN hyperparameters and overfitting (missed Q6)
KNN uses nearby training examples for predictions. A very small `k` can follow noise (high variance); a very large `k` can over-smooth the decision boundary (high bias). **Neither training accuracy nor the smallest `k` is the selection rule.** Choose among candidates using suitable validation or cross-validation results. Distance weighting and distance metrics are additional hyperparameters.

KNN also depends on scale: numeric columns with large magnitudes can dominate Euclidean distance. Use a `Pipeline` so `StandardScaler` is fitted within the training process.

In [ ]:
k_results = []
for k in [1, 3, 5, 9, 15, 31, 75]:
    knn = Pipeline([('scale', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=k))])
    knn.fit(X_train, y_train)
    k_results.append({'k': k,
                      'train_acc': accuracy_score(y_train, knn.predict(X_train)),
                      'val_acc': accuracy_score(y_val, knn.predict(X_val))})
knn_scores = pd.DataFrame(k_results)
print(knn_scores.to_string(index=False))
assert knn_scores['val_acc'].between(0, 1).all()
ax = knn_scores.plot(x='k', y=['train_acc', 'val_acc'], marker='o', figsize=(6, 4))
ax.set(title='KNN: training vs. validation accuracy', ylabel='Accuracy', xlabel='Number of neighbors (k)')
plt.show()

### Exercise 3 — pick `k` and defend the trade-off
**TODO:** select `k` using the **validation** column (not the training column); compare `weights='uniform'` and `weights='distance'`; articulate the bias/variance consequences of your choice.

In [ ]:
# TODO: Choose k based on the highest observed validation accuracy.
selected_k = None
# selected_k = int(knn_scores.loc[knn_scores['val_acc'].idxmax(), 'k'])
# assert selected_k in knn_scores['k'].values

# TODO: Fit and compare two KNN pipelines with weights='uniform' and weights='distance'.
# TODO: Explain why k=1 can have high training accuracy but weak generalization.
print('After choosing selected_k, justify it using the validation results.')

## 4. Prevent preprocessing data leakage (missed Q9)
A scaler estimates statistics such as the mean and standard deviation. Fitting it **before** splitting exposes information about validation/test feature distributions to the training pipeline. Fit on `X_train` and **transform** validation/test with the same fitted object. Do not fit another scaler to test data. When cross-validating, put the scaler **inside** a scikit-learn `Pipeline` so it is refitted on each training fold.

**Realistic spending example:** transaction totals are much larger in magnitude than login frequency; correct scaling helps distance-based models without contaminating the held-out evaluation.

In [ ]:
accounts = pd.DataFrame({
    'monthly_spend': [35, 60, 85, 110, 145, 170, 210, 240, 265, 300],
    'login_count': [2, 4, 5, 9, 12, 15, 21, 24, 25, 31],
})
train_accounts = accounts.iloc[:8].copy()
test_accounts = accounts.iloc[8:].copy()
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_accounts)
test_scaled = scaler.transform(test_accounts)
print('Training means:', np.round(scaler.mean_, 2))
print('Scaled training means:', np.round(train_scaled.mean(axis=0), 8))
print('Scaled test means:', np.round(test_scaled.mean(axis=0), 3))
assert np.allclose(train_scaled.mean(axis=0), 0, atol=1e-10)
assert np.allclose(scaler.mean_, train_accounts.mean().to_numpy())

### Exercise 4 — identify and fix leakage
**TODO:** rewrite the bad example with a train-only fit. Check what happens if the test set has a substantially different spending distribution. Then explain why two independent fits destroy the shared distance coordinate system.

In [ ]:
# BAD (do not use to evaluate a model):
# bad_scaled = StandardScaler().fit_transform(accounts)
# bad_train, bad_test = bad_scaled[:8], bad_scaled[8:]

# TODO: Use only train_accounts to fit a fresh scaler and transform test_accounts.
# clean_scaler = ...
# clean_train = ...
# clean_test = ...
# assert np.allclose(clean_scaler.mean_, train_accounts.mean().to_numpy())

# TODO: State in Markdown why fit_transform(test_accounts) is not appropriate.
print('Add your safe pipeline and inspect scaler.mean_.')

## 5. Logistic regression probability thresholds (missed Q12)
Binary logistic regression outputs **probabilities**; the decision threshold converts probabilities into classes. Lowering a threshold can create more positive predictions and often increases recall, **but it does not necessarily lower precision**. Precision depends on whether the additional predicted positives are truly positive. Threshold changes do **not** retrain model coefficients or change predicted probabilities.

**Retention outreach example:** customers are flagged for intervention when predicted churn probability is at least the threshold. Observe which confusion-matrix counts change.

In [ ]:
customers = pd.DataFrame({
    'customer': list('ABCDE'),
    'churn_probability': [0.85, 0.65, 0.45, 0.30, 0.10],
    'actual_churn': [1, 1, 1, 0, 0],
})
def evaluate_threshold(frame, threshold):
    actual = frame['actual_churn'].to_numpy()
    predictions = (frame['churn_probability'].to_numpy() >= threshold).astype(int)
    return {'threshold': threshold,
            'predicted_positive': int(predictions.sum()),
            'precision': precision_score(actual, predictions, zero_division=0),
            'recall': recall_score(actual, predictions, zero_division=0),
            'f1': f1_score(actual, predictions, zero_division=0),
            'confusion_matrix': confusion_matrix(actual, predictions, labels=[0, 1]).tolist()}
for threshold in [0.50, 0.40]:
    print(evaluate_threshold(customers, threshold))
assert evaluate_threshold(customers, 0.5)['predicted_positive'] == 2
assert evaluate_threshold(customers, 0.4)['predicted_positive'] == 3
assert evaluate_threshold(customers, 0.4)['recall'] == 1.0

### Exercise 5 — threshold selection with business costs
**TODO:** test thresholds 0.2, 0.4, 0.5, and 0.7. For each, compute TP/FP/FN/TN. Suppose missing a churner costs $120 and contacting a non-churner costs $15; determine which threshold minimizes this simple error cost on the example (then describe why you should validate that choice on a larger sample). Note that costs and thresholds are illustrative, not a calibrated production decision.

In [ ]:
# TODO: Fill the table below using evaluate_threshold and the confusion matrix.
thresholds = [0.2, 0.4, 0.5, 0.7]
loss_per_false_negative = 120
loss_per_false_positive = 15
cost_table = []
for t in thresholds:
    report = evaluate_threshold(customers, t)
    tn, fp, fn, tp = np.array(report['confusion_matrix']).ravel()
    cost = fn * loss_per_false_negative + fp * loss_per_false_positive
    cost_table.append({'threshold': t, 'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp, 'cost': cost})
print(pd.DataFrame(cost_table).to_string(index=False))
# TODO: Extract the minimum-cost threshold and explain the choice.
# TODO: Why can precision stay unchanged or even increase when a threshold decreases?
assert all(row['cost'] >= 0 for row in cost_table)

## 6. Interview-style checkpoints (without answer letters)
Explain each situation aloud or in Markdown in two to four sentences:

1. **Retail segmentation:** No labels exist. Why would you avoid calling an inferred cluster a verified class?
2. **Validation protocol:** A colleague chooses the best model based on the test set after trying 30 configurations. What went wrong?
3. **KNN model choice:** `k=1` gets 100% training accuracy, but another `k` has better validation accuracy. What would you deploy and what further checks matter?
4. **Leakage:** Why does calling `StandardScaler().fit_transform(X_all)` before the split bias evaluation?
5. **Churn policy:** Why does changing the probability threshold affect recall without changing the learned model weights?

### Sanity-check checklist
- [ ] All training, validation, and test sizes sum to the dataset size.
- [ ] Chosen KNN hyperparameters depend on validation results, not test results.
- [ ] Preprocessing statistics come only from training records.
- [ ] A new threshold changes only class predictions, not model probabilities.
- [ ] A cluster ID is not treated as an observed ground-truth target.

### Final reflection
Write **150–200 words**: **What did you learn from your missed interview questions?** Identify one assumption you would no longer make, one validation or preprocessing habit you will adopt, and one trade-off you will explain more clearly in a future ML Specialist interview.

**Optional extension:** Turn your strongest TODO solution into a reusable function and verify it against a trusted scikit-learn counterpart, following the logistic-regression workshop's manual-implementation-and-verification pattern.